# TICKET MJD-603
De: Elena Pérez · Responsable de datos
Para el Bloque 2 vamos a subir gold a Fabric, y allí se trabaja con Parquet. Exporta las siete... no, per‐
dón: las seis tablas de gold a Parquet y demuéstrame que no se ha perdido nada por el camino.

CRITERIOS
- Un fichero Parquet por tabla de gold en datos/exportaciones/gold/.
- Comprobación de filas y de una suma de control por tabla, comparando la base con el fichero.

In [8]:
import os, sys
from pathlib import Path

RAIZ = Path.home() / "master-data-ia"
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [5]:
from pathlib import Path
import duckdb
destino = Path("datos/exportaciones/gold")
destino.mkdir(parents=True, exist_ok=True)
con = duckdb.connect("db/marejada.duckdb", read_only=True)
tablas = [t for (t,) in con.sql("""
SELECT table_name FROM information_schema.tables
WHERE table_schema = 'gold' ORDER BY table_name
""").fetchall()]
print(tablas)
for t in tablas:
    con.sql(f"SELECT * FROM gold.{t}").df().to_parquet(destino / f"{t}.parquet", index=False)
con.close()
sorted(f.name for f in destino.glob("*.parquet"))

['dim_cliente', 'dim_comercial', 'dim_fecha', 'dim_producto', 'fact_pedidos', 'fact_ventas']


['dim_cliente.parquet',
 'dim_comercial.parquet',
 'dim_fecha.parquet',
 'dim_producto.parquet',
 'fact_pedidos.parquet',
 'fact_ventas.parquet']

Para la comprobación, la segunda vía es leer los ficheros con otra herramienta: DuckDB, sin pasar por pan‐
das. Por cada tabla, las filas y una suma de control de la primera columna (la clave):

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect("db/marejada.duckdb", read_only=True)
filas = []
for t in tablas:
    clave = con.sql(f"SELECT * FROM gold.{t} LIMIT 0").columns[0]
    consulta = f"SELECT count(*), sum({clave}) FROM "
    base = con.sql(consulta + f"gold.{t}").fetchone()
    fichero = con.sql(consulta + f"'{destino / (t + '.parquet')}'").fetchone()
    filas.append((t, clave, *base, *fichero))
con.close()

control = pd.DataFrame(filas, columns=["tabla", "clave", "filas_base", "suma_base", "filas_parquet",
"suma_parquet"])
control["ok"] = (control["filas_base"] == control["filas_parquet"]) & (control["suma_base"] ==
control["suma_parquet"])
control

,tabla,clave,filas_base,suma_base,filas_parquet,suma_parquet,ok
0,dim_cliente,cliente_key,1349,909225,1349,909225,True
1,dim_comercial,comercial_key,15,104,15,104,True
2,dim_fecha,fecha_key,1096,22194722043,1096,22194722043,True
3,dim_producto,producto_key,236,27729,236,27729,True
4,fact_pedidos,pedido_id,77448,156834322518410,77448,156834322518410,True
5,fact_ventas,pedido_id,366928,743038294681089,366928,743038294681089,True
